# Does Laya know when it is wrong?

Laya is a 421M decision model that publishes accuracy against TypeSafe's Jev. Accuracy tells you how often
a model is right. It does not tell you **which** answers you can trust. This notebook checks both, on the
same public datasets Laya uses, from the per-case predictions saved in `data/`.

Runs on any laptop in under a minute. No GPU, no model download.

**What is measured**
- **Accuracy**, with a 95% interval, next to the accuracy Laya publishes.
- **Does the confidence separate right answers from wrong ones?** For each system, its confidence (top
  probability) is used to sort its own answers. AUROC is the chance that a random right answer gets more
  confidence than a random wrong one (0.5 = no better than a coin). FPR at 95% recall is the share of wrong
  answers you still accept if you set the threshold to keep 95% of the right ones (lower is better).
- **Can it tell when no option is right?** 800 support tickets with 8 queues to choose from, against 800
  tickets whose true queue was removed from the list. A good system is less confident on the second group.

**Systems**: Laya (its recommended `Router`), Qwen3-8B (4-bit), Mistral-7B-Instruct v0.3, and a plain
similarity baseline (multilingual-e5-small cosine between the text and each option).

In [ ]:
import json, math
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, roc_curve
from matplotlib.patches import Rectangle
from matplotlib.lines import Line2D
DATA = "../data/"
FIG = "../figures/"
BOOT = 1000
rng = np.random.default_rng(13)

V2 = pd.read_csv(DATA + "predictions_v2.csv")                        # laya, qwen8b, similarity
V2["p"] = V2.p.map(lambda s: np.array(json.loads(s)))
MI = pd.read_csv(DATA + "predictions_mistral_and_laya_scores.csv")   # mistral, and laya raw scores

# What Laya's own repository says about each dataset (research/scripts/bench_apps.py, BENCHMARKS.md, README)
LAYA_SAW = {"ag_news": "in Laya's training mix", "support_triage": "in Laya's training mix",
            "triage_8opt": "in Laya's training mix", "no_right_answer": "in Laya's training mix",
            "emotion": "held out", "jailbreak": "held out", "jailbreak_v2": "held out",
            "massive_es": "not stated", "massive_km": "not stated"}
PUBLISHED = {"ag_news": {"laya": 0.950, "jev": 0.910}, "emotion": {"laya": 0.595, "jev": 0.480},
             "support_triage": {"laya": 0.502, "jev": None}}
SUITES = ["ag_news", "emotion", "support_triage", "triage_8opt", "jailbreak", "jailbreak_v2", "massive_es", "massive_km"]

## 1. Per-case outcome and confidence for every system

In [ ]:
def outcome(system, suite):
    """correct (0/1) and confidence (top probability) per case."""
    if system == "mistral":
        d = MI[MI.suite == suite].sort_values("i")
        return d.mistral_correct.astype(int).values, d.mistral_msp.values
    d = V2[(V2.suite == suite) & (V2.system == system)].sort_values("i")
    if d.empty: return None
    pred = d.p.map(np.argmax).values
    return (pred == d.gold.values).astype(int), d.p.map(np.max).values

def wilson(k, n, z=1.96):
    p = k / n; den = 1 + z * z / n; c = (p + z * z / (2 * n)) / den
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / den
    return c - h, c + h
def fpr95(y, s):
    f, t, _ = roc_curve(y, s); return float(f[np.searchsorted(t, 0.95)])
def boot(y, s):
    A, F = [], []
    for _ in range(BOOT):
        i = rng.integers(0, len(y), len(y))
        if 0 < y[i].mean() < 1: A.append(roc_auc_score(y[i], s[i])); F.append(fpr95(y[i], s[i]))
    return np.percentile(A, [2.5, 97.5]), np.percentile(F, [2.5, 97.5])

rows = []
for suite in SUITES:
    for system in ["laya", "qwen8b", "mistral", "similarity"]:
        o = outcome(system, suite)
        if o is None: continue
        y, s = o; lo, hi = wilson(y.sum(), len(y)); (alo, ahi), (flo, fhi) = boot(y, s)
        rows.append({"suite": suite, "laya_saw_data": LAYA_SAW[suite], "system": system, "n": len(y),
                     "accuracy": y.mean(), "acc_lo": lo, "acc_hi": hi,
                     "auroc": roc_auc_score(y, s), "auroc_lo": alo, "auroc_hi": ahi,
                     "fpr95": fpr95(y, s), "fpr95_lo": flo, "fpr95_hi": fhi})

# no right answer: answerable tickets (triage_8opt) vs tickets whose queue was removed, same 8 options
def nra_scores(system):
    if system == "mistral":
        a = MI[MI.suite == "triage_8opt"].mistral_msp.values; b = MI[MI.suite == "no_right_answer"].mistral_msp.values
    else:
        a = outcome(system, "triage_8opt")[1]
        b = V2[(V2.suite == "no_right_answer") & (V2.system == system)].sort_values("i").p.map(np.max).values
    return np.r_[np.ones(len(a)), np.zeros(len(b))].astype(int), np.r_[a, b]
for system in ["laya", "qwen8b", "mistral", "similarity"]:
    y, s = nra_scores(system); (alo, ahi), (flo, fhi) = boot(y, s)
    rows.append({"suite": "no_right_answer", "laya_saw_data": LAYA_SAW["no_right_answer"], "system": system, "n": len(y),
                 "accuracy": np.nan, "auroc": roc_auc_score(y, s), "auroc_lo": alo, "auroc_hi": ahi,
                 "fpr95": fpr95(y, s), "fpr95_lo": flo, "fpr95_hi": fhi})
RES = pd.DataFrame(rows)
RES.to_csv("../results.csv", index=False)
pd.set_option("display.width", 200)
print(RES[["suite", "laya_saw_data", "system", "accuracy", "auroc", "fpr95"]].round(3).to_string(index=False))

## 2. Accuracy: what Laya publishes vs what we measured

In [ ]:
plt.rcParams.update({"font.family": "sans-serif", "font.sans-serif": ["Helvetica", "Arial", "Liberation Sans", "DejaVu Sans"],
                     "axes.spines.top": False, "axes.spines.right": False, "axes.edgecolor": "#9A9A9A",
                     "axes.labelcolor": "#555555", "xtick.color": "#9A9A9A", "ytick.color": "#9A9A9A",
                     "xtick.labelcolor": "#777777", "ytick.labelcolor": "#777777", "axes.titlelocation": "left",
                     "axes.titleweight": "bold", "savefig.dpi": 300, "savefig.bbox": "tight"})
INK, ORG, GRY, BLU, LGT, MUTED = "#333333", "#F0571F", "#7A7A7A", "#4C78A8", "#B5B5B5", "#777777"
NAME = {"ag_news": "AG News\n4 labels", "emotion": "DAIR Emotion\n6 labels", "support_triage": "Support triage\n9-10 queues"}
fig, ax = plt.subplots(figsize=(7.2, 4.4))
for k, s in enumerate(NAME):
    r = RES[(RES.suite == s) & (RES.system == "laya")].iloc[0]
    ax.scatter(k - 0.24, PUBLISHED[s]["laya"], s=60, facecolor="white", edgecolor=INK, lw=1.6, zorder=3)
    ax.plot([k - 0.12] * 2, [r.acc_lo, r.acc_hi], color=INK, lw=1.2); ax.scatter(k - 0.12, r.accuracy, s=60, color=INK, zorder=3)
    if PUBLISHED[s]["jev"]: ax.scatter(k, PUBLISHED[s]["jev"], s=70, marker="D", color=ORG, zorder=3)
    else: ax.text(k, 0.06, "Jev\nnot\nreported", ha="center", va="bottom", fontsize=7, color=ORG)
    for dx, sy, col in [(0.12, "qwen8b", GRY), (0.24, "mistral", BLU)]:
        q = RES[(RES.suite == s) & (RES.system == sy)].iloc[0]
        ax.plot([k + dx] * 2, [q.acc_lo, q.acc_hi], color=col, lw=1); ax.scatter(k + dx, q.accuracy, s=40, color=col, zorder=3)
ax.set_xticks(range(3)); ax.set_xticklabels(NAME.values()); ax.set_xlim(-0.5, 2.5); ax.set_ylim(0, 1.02); ax.set_ylabel("accuracy")
ax.set_title("Laya and Jev: published accuracy vs what we measured")
ax.legend(handles=[Line2D([], [], marker="o", ls="", mfc="white", mec=INK, mew=1.6, ms=7, label="Laya, published"),
                   Line2D([], [], marker="o", ls="", color=INK, ms=7, label="Laya, measured (95% interval)"),
                   Line2D([], [], marker="D", ls="", color=ORG, ms=7, label="Jev, published"),
                   Line2D([], [], marker="o", ls="", color=GRY, ms=6, label="Qwen3-8B, measured"),
                   Line2D([], [], marker="o", ls="", color=BLU, ms=6, label="Mistral-7B, measured")], loc="upper right", fontsize=8, frameon=False)
fig.text(0.01, -0.06, "Published: Laya BENCHMARKS.md (400-case subsets; Jev figures are third-party and called indicative there). Measured: 800 cases, checkpoint laya.\n"
         "Support triage: Laya publishes a 10-way version; ours has 9 queues for the comparison panel. Jev publishes accuracy and calibration only.", fontsize=7, color=MUTED)
plt.savefig(FIG + "accuracy_published_vs_measured.png"); plt.show()

## 3. Does each system know when it is wrong?

Background zones (same in every chart):
- **green**: AUROC above 0.7 and FPR below 0.5. Confidence you can act on.
- **orange**: AUROC above 0.7 but FPR above 0.5. Confidence ranks answers, but at a safe threshold too many errors still pass.
- **red**: AUROC below 0.7. Confidence tells you little.

In [ ]:
C = {"laya": "#1F4E9E", "qwen8b": "#8E3BB0", "mistral": "#E0115F", "similarity": "#00A39A"}
LAB = {"laya": "Laya", "qwen8b": "Qwen3-8B", "mistral": "Mistral-7B", "similarity": "Similarity (e5-small)"}
ORDER = ["similarity", "qwen8b", "mistral", "laya"]
T = {"ag_news": "AG News", "emotion": "Emotion", "support_triage": "Support triage", "triage_8opt": "Triage, 8 queues",
     "jailbreak": "Jailbreak", "jailbreak_v2": "Jailbreak v2", "massive_es": "MASSIVE Spanish", "massive_km": "MASSIVE Khmer",
     "no_right_answer": "No right answer in the list"}
TAG = {"in Laya's training mix": "in Laya's training", "held out": "held out", "not stated": ""}
Y0, YM, XM = 0.4, 0.7, 0.5
def zones(ax, labels=False):
    ax.add_patch(Rectangle((0, YM), XM, 1 - YM, facecolor="#2E9E4F", alpha=0.22, lw=0, zorder=0))     # green
    ax.add_patch(Rectangle((XM, YM), 1 - XM, 1 - YM, facecolor="#F28C28", alpha=0.24, lw=0, zorder=0))  # orange
    ax.add_patch(Rectangle((0, Y0), 1, YM - Y0, facecolor="#D7263D", alpha=0.16, lw=0, zorder=0))      # red
    ax.axhline(0.5, color="#9A9A9A", lw=0.8, ls="--", zorder=1)
    if labels:
        kw = dict(fontsize=8.5, fontweight="bold", alpha=0.85, zorder=1)
        ax.text(0.02, 0.985, "confidence you can act on", color="#1E7A3A", va="top", **kw)
        ax.text(0.98, 0.985, "ranks answers, but too many\nerrors pass at a safe threshold", color="#B25E0C", va="top", ha="right", **kw)
        ax.text(0.98, Y0 + 0.012, "confidence tells you little", color="#A31328", va="bottom", ha="right", **kw)
        ax.text(0.02, 0.505, "coin toss", color="#777777", fontsize=7.5, va="bottom")
legend_handles = [Line2D([], [], marker="o", ls="", color=C[k], ms=8, label=LAB[k]) for k in ["laya", "qwen8b", "mistral", "similarity"]]

### 3a. All datasets at once: the average of the nine

Small points: each system on each dataset. Large points: its average over the nine (similarity has no
jailbreak runs, so its average is over seven).

In [ ]:
fig, ax = plt.subplots(figsize=(7.6, 6.2))
zones(ax, labels=True)
avg = RES.groupby("system")[["auroc", "fpr95"]].mean()
for system in ORDER:
    d = RES[RES.system == system]
    ax.scatter(d.fpr95, d.auroc, s=22, color=C[system], alpha=0.35, edgecolor="none", zorder=2)
    m = avg.loc[system]
    ax.scatter(m.fpr95, m.auroc, s=190, color=C[system], edgecolor="white", lw=2, zorder=4)
LABEL_Y = {"laya": 0.712, "mistral": 0.678, "qwen8b": 0.644, "similarity": 0.610}   # labels stacked on the left, joined by a thin line
for system in ["laya", "mistral", "qwen8b", "similarity"]:
    m = avg.loc[system]
    ax.annotate(f"{LAB[system]}   AUROC {m.auroc:.2f}, FPR {m.fpr95:.2f}", (m.fpr95, m.auroc), xytext=(0.62, LABEL_Y[system]),
                fontsize=9, color=C[system], fontweight="bold", ha="right", va="center", zorder=5,
                arrowprops=dict(arrowstyle="-", color=C[system], lw=0.9, shrinkA=2, shrinkB=7))
ax.set_xlim(0, 1); ax.set_ylim(Y0, 1.0)
ax.set_xlabel("FPR at 95% recall (lower is better)"); ax.set_ylabel("AUROC (higher is better)")
ax.set_title("Does each system know when it is wrong? Average of nine datasets")
fig.text(0.01, -0.02, "Large points: average over the nine datasets (similarity: seven, no jailbreak runs). Small points: each dataset.\n"
         "AUROC: chance that a right answer gets more confidence than a wrong one. FPR at 95% recall: share of errors that pass\n"
         "when the threshold keeps 95% of the right answers.", fontsize=7, color=MUTED)
plt.savefig(FIG + "summary_auroc_vs_fpr95.png"); plt.show()
print(avg.round(3))

### 3b. Dataset by dataset

In [ ]:
fig, axs = plt.subplots(3, 3, figsize=(10, 10.2), sharex=True, sharey=True)
for ax, s in zip(axs.flat, T):
    zones(ax)
    for system in ORDER:
        d = RES[(RES.suite == s) & (RES.system == system)]
        if d.empty: continue
        r = d.iloc[0]
        ax.plot([r.fpr95] * 2, [max(r.auroc_lo, Y0), r.auroc_hi], color=C[system], lw=1.2, alpha=0.7, zorder=2)
        ax.plot([r.fpr95_lo, r.fpr95_hi], [r.auroc] * 2, color=C[system], lw=1.2, alpha=0.7, zorder=2)
        ax.scatter(r.fpr95, max(r.auroc, Y0), s=70, color=C[system], edgecolor="white", lw=1.3, zorder=3)
    ax.set_title(T[s], fontsize=10)
    if TAG[LAYA_SAW[s]]: ax.text(0.02, 0.03, TAG[LAYA_SAW[s]], transform=ax.transAxes, fontsize=7.5, color="#555555")
ax.set_xlim(0, 1); ax.set_ylim(Y0, 1.0)
for a in axs[-1]: a.set_xlabel("FPR at 95% recall (lower is better)")
for a in axs[:, 0]: a.set_ylabel("AUROC (higher is better)")
fig.legend(handles=legend_handles, loc="upper left", bbox_to_anchor=(0.01, 0.965), ncol=4, fontsize=9, frameon=False)
fig.suptitle("Does each system know when it is wrong? Dataset by dataset", x=0.01, ha="left", fontsize=12, fontweight="bold")
fig.text(0.01, 0.0, "First eight panels: how well each system's confidence separates its own right answers from its errors (800 cases; jailbreak 408).\n"
         "Last panel: 800 answerable tickets vs 800 tickets whose true queue was removed from the list. Bars: 95% bootstrap intervals.\n"
         "Green: confidence you can act on. Orange: it ranks answers but too many errors pass at a safe threshold. Red: it tells you little.", fontsize=7, color=MUTED)
plt.tight_layout(rect=(0, 0.045, 1, 0.94)); plt.savefig(FIG + "auroc_vs_fpr95.png"); plt.show()

## 4. Summary in numbers

In [ ]:
la = RES[RES.system == "laya"].set_index("suite")
print("Laya, datasets it says it trained on vs held out:")
for grp in ["in Laya's training mix", "held out", "not stated"]:
    sub = la[la.laya_saw_data == grp]
    print(f"  {grp:24s} AUROC {', '.join(f'{s} {v:.2f}' for s, v in sub.auroc.items())}")
best = RES.loc[RES.groupby("suite").auroc.idxmax()][["suite", "system", "auroc", "fpr95"]]
print("\nBest AUROC per dataset:\n", best.round(3).to_string(index=False))